In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("innocentmfa/crm-sales-opportunities")

print("Path to dataset files:", path)

100%|██████████| 143k/143k [00:00<00:00, 1.49MB/s]

Extracting files...


Path to dataset files: C:\Users\jaure\.cache\kagglehub\datasets\innocentmfa\crm-sales-opportunities\versions\1


In [2]:
import os
print(os.listdir(path))

['accounts.csv', 'data_dictionary.csv', 'products.csv', 'sales_pipeline.csv', 'sales_teams.csv']


In [3]:
import pandas as pd

accounts = pd.read_csv(os.path.join(path, "accounts.csv"))
pipeline = pd.read_csv(os.path.join(path, "sales_pipeline.csv"))

accounts.head()

,account,sector,year_established,revenue,employees,office_location,subsidiary_of
0,Acme Corporation,technolgy,1996,1100.04,2822,United States,NaN
1,Betasoloin,medical,1999,251.41,495,United States,NaN
2,Betatech,medical,1986,647.18,1185,Kenya,NaN
3,Bioholding,medical,2012,587.34,1356,Philipines,NaN
4,Bioplex,medical,1991,326.82,1016,United States,NaN


In [4]:
import shutil, os

os.makedirs("data/raw", exist_ok=True)
for f in ["accounts.csv", "sales_pipeline.csv"]:
    shutil.copy(os.path.join(path, f), "data/raw/" + f)

In [5]:
print(accounts.shape)
print(accounts["account"].duplicated().sum())
print(accounts["sector"].value_counts())
print(accounts["office_location"].value_counts())

(85, 7)
0
sector
retail                17
technolgy             12
medical               12
marketing              8
finance                8
software               7
entertainment          6
telecommunications     6
services               5
employment             4
Name: count, dtype: int64
office_location
United States    71
Kenya             1
Philipines        1
Japan             1
Italy             1
Norway            1
Korea             1
Jordan            1
Brazil            1
Germany           1
Panama            1
Belgium           1
Romania           1
Poland            1
China             1
Name: count, dtype: int64


In [7]:
pipeline.head()

,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value
0,1C1I7A6R,Moses Frase,GTX Plus Basic,Cancity,Won,2016-10-20,2017-03-01,1054.0
1,Z063OYW0,Darcel Schlecht,GTXPro,Isdom,Won,2016-10-25,2017-03-11,4514.0
2,EC4QE1BX,Darcel Schlecht,MG Special,Cancity,Won,2016-10-25,2017-03-07,50.0
3,MV1LWRNH,Moses Frase,GTX Basic,Codehow,Won,2016-10-25,2017-03-09,588.0
4,PE84CX4O,Zane Levy,GTX Basic,Hatfan,Won,2016-10-25,2017-03-02,517.0


In [6]:
print(pipeline.shape)
print(pipeline["deal_stage"].value_counts())
print(pipeline.isna().sum())

(8800, 8)
deal_stage
Won            4238
Lost           2473
Engaging       1589
Prospecting     500
Name: count, dtype: int64
opportunity_id       0
sales_agent          0
product              0
account           1425
deal_stage           0
engage_date        500
close_date        2089
close_value       2089
dtype: int64


In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("data/raw")
OUT = Path("data/salesforce_ready")
OUT.mkdir(parents=True, exist_ok=True)

accounts = pd.read_csv(RAW / "accounts.csv")
pipeline = pd.read_csv(RAW / "sales_pipeline.csv")

# ---------- Cuentas ----------
# No se corrigen 'technolgy' ni 'Philipines' a propósito: los limpiará PySpark.
sf_accounts = pd.DataFrame({
    "Name": accounts["account"],
    "Industry": accounts["sector"],
    "BillingCountry": accounts["office_location"],
    "AnnualRevenue": (accounts["revenue"] * 1_000_000).round(0),
    "NumberOfEmployees": accounts["employees"].astype(int),
})

# ---------- Oportunidades ----------
stage_map = {
    "Prospecting": "Prospecting",
    "Engaging": "Qualification",
    "Won": "Closed Won",
    "Lost": "Closed Lost",
}
fallback_date = pd.Timestamp.today().normalize() + pd.Timedelta(days=90)

close_dates = pd.to_datetime(pipeline["close_date"]).fillna(fallback_date)

sf_opps = pd.DataFrame({
    "Name": pipeline["product"] + " - " + pipeline["account"].fillna("Sin cuenta"),
    "AccountName": pipeline["account"],   # temporal; el script lo cambia por AccountId
    "StageName": pipeline["deal_stage"].map(stage_map),
    "CloseDate": close_dates.dt.strftime("%Y-%m-%d"),
    "Amount": pipeline["close_value"],
    # Guarda el id original y datos extra sin crear campos personalizados
    "Description": (
        "opportunity_id=" + pipeline["opportunity_id"].astype(str)
        + "; sales_agent=" + pipeline["sales_agent"].astype(str)
        + "; engage_date=" + pipeline["engage_date"].fillna("").astype(str)
    ),
})

# ---------- Comprobaciones ----------
assert sf_accounts["Name"].is_unique, "Hay nombres de cuenta repetidos"
assert sf_opps["StageName"].notna().all(), "Hay etapas sin traducir"
desconocidas = set(pipeline["account"].dropna()) - set(accounts["account"])
assert not desconocidas, f"Cuentas inexistentes: {desconocidas}"

sf_accounts.to_csv(OUT / "accounts_sf.csv", index=False)
sf_opps.to_csv(OUT / "opportunities_sf.csv", index=False)

print(sf_accounts.shape, sf_opps.shape)
print("Oportunidades sin cuenta:", sf_opps["AccountName"].isna().sum())
sf_opps.head()

(85, 5) (8800, 6)
Oportunidades sin cuenta: 1425


,Name,AccountName,StageName,CloseDate,Amount,Description
0,GTX Plus Basic - Cancity,Cancity,Closed Won,2017-03-01,1054.0,opportunity_id=1C1I7A6R; sales_agent=Moses Fra...
1,GTXPro - Isdom,Isdom,Closed Won,2017-03-11,4514.0,opportunity_id=Z063OYW0; sales_agent=Darcel Sc...
2,MG Special - Cancity,Cancity,Closed Won,2017-03-07,50.0,opportunity_id=EC4QE1BX; sales_agent=Darcel Sc...
3,GTX Basic - Codehow,Codehow,Closed Won,2017-03-09,588.0,opportunity_id=MV1LWRNH; sales_agent=Moses Fra...
4,GTX Basic - Hatfan,Hatfan,Closed Won,2017-03-02,517.0,opportunity_id=PE84CX4O; sales_agent=Zane Levy...
